# 09 — Basis Extension and BIC Model Selection

The standard 15-coefficient distortion basis is enough for most
synchrotron area detectors at moderate 2θ.  When the
held-out-ring CV gate fires (notebook **05**), one option is the
F2 per-ring offset; another is **extending the harmonic basis**
to higher η-folds.

This notebook shows:
1. How the 15-coef basis is defined as a list of `HarmonicTerm`
   tuples.
2. How to extend to fold-7+ via `extended_term_layout(max_fold=N)`.
3. How **BIC** (Bayesian Information Criterion) decides whether
   the extra parameters are justified by the data.


In [1]:
import os, math
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
from pathlib import Path
import numpy as np
import torch
from PIL import Image

from midas_calibrate_v2.forward.distortion import (
    P_COEF_NAMES, v2_term_layout,
    extended_term_layout, extended_p_coef_names,
)

# Show the standard basis
print('Standard 15-coef basis:')
for term in v2_term_layout():
    print(f'  fold={term.fold:>2d}  power=ρ^{term.radial_power}  '
          f'coef_idx={term.coef_idx:>2d}  phase_idx={term.phase_idx:>2d}')

print(f'\nNamed coefficients: {P_COEF_NAMES}')


Standard 15-coef basis:
  fold= 0  power=ρ^2  coef_idx= 0  phase_idx=-1
  fold= 0  power=ρ^4  coef_idx= 1  phase_idx=-1
  fold= 0  power=ρ^6  coef_idx= 2  phase_idx=-1
  fold= 1  power=ρ^4  coef_idx= 3  phase_idx= 4
  fold= 2  power=ρ^2  coef_idx= 5  phase_idx= 6
  fold= 3  power=ρ^3  coef_idx= 7  phase_idx= 8
  fold= 4  power=ρ^4  coef_idx= 9  phase_idx=10
  fold= 5  power=ρ^5  coef_idx=11  phase_idx=12
  fold= 6  power=ρ^6  coef_idx=13  phase_idx=14

Named coefficients: ['iso_R2', 'iso_R4', 'iso_R6', 'a1', 'phi1', 'a2', 'phi2', 'a3', 'phi3', 'a4', 'phi4', 'a5', 'phi5', 'a6', 'phi6']


## Extend to fold-8

Adds amplitude/phase pairs `(a7, phi7)`, `(a8, phi8)` at radial
powers ρ⁷, ρ⁸.


In [2]:
extended = extended_term_layout(max_fold=8)
extended_names = extended_p_coef_names(max_fold=8)
print(f'Extended basis to max_fold=8: {len(extended)} terms')
for term in extended[-4:]:
    print(f'  fold={term.fold:>2d}  power=ρ^{term.radial_power}  '
          f'coef_idx={term.coef_idx:>2d}  phase_idx={term.phase_idx:>2d}')
print(f'\nExtended names: {extended_names}')


Extended basis to max_fold=8: 11 terms
  fold= 5  power=ρ^5  coef_idx=11  phase_idx=12
  fold= 6  power=ρ^6  coef_idx=13  phase_idx=14
  fold= 7  power=ρ^7  coef_idx=15  phase_idx=16
  fold= 8  power=ρ^8  coef_idx=17  phase_idx=18

Extended names: ['iso_R2', 'iso_R4', 'iso_R6', 'a1', 'phi1', 'a2', 'phi2', 'a3', 'phi3', 'a4', 'phi4', 'a5', 'phi5', 'a6', 'phi6', 'a7', 'phi7', 'a8', 'phi8']


## BIC for model selection

The BIC penalises models for free parameters:

  BIC = N · log(RSS / N) + k · log(N)

A larger model is preferred only if BIC decreases.  Below: a quick
illustrative computation comparing the 15-coef baseline against
extended fold-7 and fold-8 fits on Varex CeO₂.

For the full paper sweep across all four reference datasets, see
[`dev/paper/runners/run_harmonic_extension.py`](../dev/paper/runners/run_harmonic_extension.py).


In [3]:
import time
from midas_calibrate.params import CalibrationParams as V1Params
from midas_calibrate_v2.compat.from_v1 import spec_from_v1_file
from midas_calibrate_v2.parameters.parameter import Parameter
from midas_calibrate_v2.pipelines.single_pv import autocalibrate_pv
from midas_calibrate_v2.loss.pseudo_strain import pseudo_strain_residual

BASE = Path(os.environ.get('V2_TEST_BASE', '/tmp/midas_v2_test'))
PARAMS = BASE / 'refined_MIDAS_params_Ceria_63keV_900mm_100x100_0p5s_aero_0.txt'
IMAGE  = BASE / 'Ceria_63keV_900mm_100x100_0p5s_aero_0_001137.tif'

v1 = V1Params.from_file(PARAMS)
if v1.RBinSize <= 0: v1.RBinSize = 0.25
if v1.EtaBinSize <= 0: v1.EtaBinSize = 5.0
v1.MaxRingRad = max(v1.MaxRingRad, v1.RhoD / max(v1.pxY, 1.0))
v1.Width = max(v1.Width, 800.0)
image = np.array(Image.open(str(IMAGE))).astype(np.float64)[::-1, :].copy()

spec_15 = spec_from_v1_file(PARAMS)
print('Running 15-coef baseline (~30 s)…')
t0 = time.time()
res_15 = autocalibrate_pv(
    v1, image, spec=spec_15,
    n_iter=4, half_window_px=8.0, snr_min=8.0,
    trim_mode='stratified_multfactor', trim_residual_pct=5.0,
    reuse_fits=True, lm_max_iter=300, verbose=False,
    distribution_report=False,
)
fits_15 = res_15.fits_final
with torch.no_grad():
    r_15 = pseudo_strain_residual(
        fits_15.Y_pix, fits_15.Z_pix, fits_15.ring_two_theta_deg, res_15.unpacked,
        rho_d=fits_15.rho_d, weights=fits_15.weights,
        ring_idx=fits_15.ring_idx, ring_d_spacing_A=fits_15.ring_d_spacing_A,
    )
N_15 = int(r_15.numel())
RSS_15 = float((r_15 * r_15).sum())
k_15 = sum(p.numel for p in spec_15.parameters.values() if p.refined)
BIC_15 = N_15 * math.log(RSS_15 / N_15) + k_15 * math.log(N_15)
print(f'  15-coef:  k={k_15}  RSS={RSS_15:.4e}  N={N_15}  BIC={BIC_15:.2f}  '
      f'strain={float(r_15.abs().mean())*1e6:.2f} µε  '
      f'({time.time()-t0:.1f}s)')


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running 15-coef baseline (~30 s)…


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  return torch.sparse_csr_tensor(indptr, indices, values,
/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(indptr, in

  15-coef:  k=20  RSS=3.2475e-06  N=5374  BIC=-113901.86  strain=18.48 µε  (26.0s)


## Why no fold-7+ run here?

A live extended-basis run requires the pipeline to instantiate
the extra `Parameter`s and propagate them through every closure.
Doing this in a notebook means either:

(a) Manually adding `a7, phi7, a8, phi8` to the spec and adapting
    the cake/peak-fit infrastructure to know about them; or

(b) Using the dedicated runner
    `dev/paper/runners/run_harmonic_extension.py` which handles
    all the wiring correctly and emits BIC for every combination
    on every reference dataset.

For pedagogical purposes, the take-away is the **conceptual recipe**:
- BIC at 15-coef is your baseline.
- Add a fold (or remove one), refit, re-evaluate BIC.
- If BIC drops, the extra parameter is justified by the data.
- If BIC rises, you're overfitting — keep the smaller basis.

The **BIC ladder** in `tab:bic_ladder` of the paper shows that on
all four reference datasets, BIC stays minimal at 15-coef across
fold-7 and fold-8 attempts — the data doesn't support extra η-folds.

## When extending the basis is the right move

- High-2θ rings show a **smooth** residual systematic that the
  CV gate flags but the per-ring DC `δr_k` doesn't fully fix.
- The extra fold's amplitude is statistically distinguishable
  from zero in the Laplace σ.
- BIC decreases (not just RSS).

If those three conditions aren't met, the F2 per-ring offset is
the better fix because it's parameterised by physics (per-(hkl)
DC) rather than by mathematical convenience (more harmonics).

## See also

- [`dev/paper/runners/run_harmonic_extension.py`](../dev/paper/runners/run_harmonic_extension.py) — the BIC ladder runner that produces `tab:bic_ladder`
- [`forward/distortion.py`](../midas_calibrate_v2/forward/distortion.py) — `HarmonicTerm`, `extended_term_layout`, `extended_p_coef_names`
